# 04 – GGUF-Export (ohne Eval zu wiederholen)
Lädt **adapter_grpo** und exportiert für Ollama / LM Studio:
- **Teil A, immer:** LoRA-Adapter als GGUF (~130 MB) + `Modelfile.lora`. Läuft auf dem Mac mit `ollama pull qwen3:14b`.
- **Teil B, wenn der Platz reicht:** Komplett-Modell `q4_k_m` (~9 GB) + `Modelfile`. Die Zwischendateien liegen in `/tmp`, nicht im 20-GB-Output.

**Optional:** Wird zusätzlich der Eval-Lauf als Input angehängt, berechnet eine Zelle die **Konfidenzintervalle** (Basis vs. trainiert).

**Kaggle:** Input = GRPO-Lauf (+ optional Eval-Lauf) · GPU T4 x2 · Internet an · Save & Run All (Commit). Dauer ~1 h.

In [ ]:
# === Setup ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"
import os, sys, glob, json, shutil, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml sentence-transformers
sys.path.insert(0, "/kaggle/working/repo/scripts")
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
EXPORT = "/kaggle/working/export"; os.makedirs(EXPORT, exist_ok=True)
FULL_GGUF = True   # False = nur LoRA-GGUF (Teil A), spart ~45 Min

def free_gb(p):
    try: return shutil.disk_usage(p).free / 1e9
    except Exception: return None
for p in ("/kaggle/working", "/tmp", os.path.expanduser("~/.cache")):
    print(f"frei {p:28s} {free_gb(p):6.1f} GB")

In [ ]:
# === Inputs finden ===
step_of = lambda p: int(p.rstrip("/").split("-")[-1])
cands = glob.glob("/kaggle/input/**/adapter_grpo/adapter_model.safetensors", recursive=True)
if not cands:   # Rückfall: neuester GRPO-Checkpoint
    ck = sorted(glob.glob("/kaggle/input/**/checkpoints_grpo/checkpoint-*/adapter_model.safetensors", recursive=True),
                key=lambda p: step_of(os.path.dirname(p)))
    cands = ck[-1:]
assert cands, "adapter_grpo nicht gefunden – rechts 'Add Input → Your Work' den GRPO-Lauf hinzufügen"
ADAPTER = cands[0]; print("Adapter:", ADAPTER)
EVAL = glob.glob("/kaggle/input/**/eval_outputs.json", recursive=True)
print("eval_outputs.json:", EVAL[0] if EVAL else "– (Signifikanz-Zelle wird übersprungen)")

In [ ]:
# === Optional: Signifikanz der Eval (nur wenn eval_outputs.json als Input da ist) ===
# Paarweiser Vergleich je Frage (Mittel über Samples), 10.000 Bootstrap-Ziehungen → 95%-Intervall.
# Originalität nur dort, wo eine faire Referenz existiert ('obvious'); bei 'allgemein' war die Referenz
# im Eval die Basis-Antwort selbst → der Wert bevorzugt 'tuned' systematisch und wird hier nicht gewertet.
import numpy as np, re, torch
if EVAL:
    from judge_client import Judge
    from sentence_transformers import SentenceTransformer
    E = CFG["eval"]; dev = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
    emb = SentenceTransformer(E["embed_model"], device=dev)
    J = Judge(E["judge_model"], gpu=1 if torch.cuda.device_count() > 1 else 0)
    JUDGE = """Problem: {q}\n\nAntwort: {a}\n\nBewerte NUR, ob die vorgeschlagene Lösung das Problem realistisch, sicher und legal löst. Originalität ist egal. Vage Floskeln höchstens 3.\nAntworte ausschließlich mit einer ganzen Zahl von 0 bis 10."""
    def final(t):
        m = re.search(r"^Beste Idee:\s*(.+)", t, re.M | re.S); return (m.group(1) if m else t[-800:]).strip()
    res = json.load(open(EVAL[0]))
    per = {}   # (kind, prompt) -> {"base": [...], "tuned": [...]}
    vals = J.raw([JUDGE.format(q=r["prompt"], a=final(r[w])[:1500]) for r in res for w in ("base", "tuned")])
    for i, r in enumerate(res):
        d = per.setdefault((r["kind"], r["prompt"]), {"ob": r.get("obvious"), "base": [], "tuned": []})
        for j, w in enumerate(("base", "tuned")):
            o = None
            if r.get("obvious"):
                a, b = emb.encode([f"query: {final(r[w])}", f"query: {r['obvious']}"], normalize_embeddings=True)
                o = float(1 - a @ b)
            d[w].append((o, vals[2 * i + j]))
    rng = np.random.default_rng(0)
    def ci(diffs):
        diffs = np.array(diffs); bs = rng.choice(diffs, (10000, len(diffs))).mean(1)
        return diffs.mean(), *np.percentile(bs, [2.5, 97.5]), (diffs > 0).mean()
    print(f"{'Gruppe':20s} {'Maß':12s} {'n':>3s}  {'Δ tuned−base':>12s}  {'95%-Intervall':>18s}  tuned besser")
    for kind in ("allgemein", "business", "business_evolution"):
        items = [v for (k, _), v in per.items() if k == kind]
        for name, idx in (("Originalität", 0), ("Gültigkeit", 1)):
            if name == "Originalität" and kind == "allgemein":
                print(f"{kind:20s} {name:12s}   –  (keine faire Referenz, siehe Kommentar)"); continue
            diffs = [np.mean([x[idx] for x in v["tuned"]]) - np.mean([x[idx] for x in v["base"]]) for v in items]
            m, lo, hi, share = ci(diffs)
            sig = "✓ signifikant" if lo > 0 or hi < 0 else "n. s."
            print(f"{kind:20s} {name:12s} {len(diffs):3d}  {m:+12.3f}  [{lo:+.3f}, {hi:+.3f}]  {share:5.0%}  {sig}")
    J.close(); del emb; torch.cuda.empty_cache()

In [ ]:
# === Modell laden: Basis + LoRA, Gewichte aus adapter_grpo (verifiziert) ===
from unsloth import FastLanguageModel
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
L = CFG["lora"]
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"], max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=L["r"], lora_alpha=L["alpha"], lora_dropout=L["dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], random_state=42)
res = set_peft_model_state_dict(model, load_file(ADAPTER))
missing = [k for k in res.missing_keys if "lora_" in k]
assert not missing and not res.unexpected_keys, "Adapter passt nicht zum Basismodell"
print("✓ Adapter geladen")

In [ ]:
# === Teil A: LoRA-Adapter als GGUF (~130 MB, sicher) – für Ollama mit qwen3:14b ===
out = model.save_pretrained_gguf("/kaggle/working/lora_gguf", tokenizer, save_method="lora", quantization_method="f16")
lora = (glob.glob("/kaggle/working/lora_gguf/**/*.gguf", recursive=True) or [out])[0]
shutil.copy(lora, f"{EXPORT}/querdenker-lora-f16.gguf")
open(f"{EXPORT}/Modelfile.lora", "w").write(
    "FROM qwen3:14b\n"
    "ADAPTER ./querdenker-lora-f16.gguf\n"
    "PARAMETER temperature 0.8\n")
print("✓ Teil A:", f"{EXPORT}/querdenker-lora-f16.gguf", f"{os.path.getsize(EXPORT + '/querdenker-lora-f16.gguf') / 1e6:.0f} MB")

In [ ]:
# === Teil B: Komplett-GGUF q4_k_m (~9 GB) – Arbeitsverzeichnis nach /tmp verlegen ===
# Ursache des Fehlers im Eval-Lauf: Unsloth schreibt die f16-Zwischendatei (~30 GB) ins aktuelle
# Arbeitsverzeichnis (/kaggle/working, max. 20 GB). Hier liegt alles Große in /tmp; nur das fertige q4_k_m
# wird nach /kaggle/working kopiert. Scheitert es trotzdem (Platz), bleibt Teil A erhalten.
if FULL_GGUF:
    WORK = "/tmp/gguf_work"; os.makedirs(WORK, exist_ok=True)
    shutil.rmtree("/kaggle/working/lora_gguf", ignore_errors=True)
    print(f"frei /tmp: {free_gb('/tmp'):.1f} GB")
    old = os.getcwd(); os.chdir(WORK)
    try:
        model.save_pretrained_gguf(os.path.join(WORK, "gguf"), tokenizer, quantization_method="q4_k_m")
        q4 = [f for f in glob.glob(f"{WORK}/**/*.gguf", recursive=True) if "q4_k_m" in f.lower()]
        assert q4, "Keine q4_k_m-Datei gefunden: " + str(glob.glob(f"{WORK}/**/*.gguf", recursive=True))
        shutil.copy(q4[0], f"{EXPORT}/querdenker-14b-q4_k_m.gguf")
        open(f"{EXPORT}/Modelfile", "w").write(
            "FROM ./querdenker-14b-q4_k_m.gguf\n"
            "PARAMETER temperature 0.8\n")
        print("✓ Teil B:", f"{EXPORT}/querdenker-14b-q4_k_m.gguf")
    except Exception as e:
        print("⚠ Teil B nicht möglich:", type(e).__name__, str(e)[:600])
        print("→ Teil A (LoRA-GGUF + Modelfile.lora) ist trotzdem im Output und reicht für Ollama.")
    finally:
        os.chdir(old); shutil.rmtree(WORK, ignore_errors=True)

In [ ]:
for f in sorted(os.listdir(EXPORT)):
    print(f"{f:36s} {os.path.getsize(os.path.join(EXPORT, f)) / 1e9:6.2f} GB")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)   # Output schlank halten
print("\nDownload: Output → export/")